# 02 The leak

A dead end, kept. The first version of the merchant category fraud rate counted every fraud the label feed ever reported for the category in the twelve months before the event. It used labels by valid time alone: a chargeback reported in March counted toward a February transaction's feature. The as of test caught it.

The test plants one label known one second after the start of the month and checks the month's rate does not move. The leaky version moved; the version in packages/transforms/gold.py (known_frauds) reads each label's system time interval and does not.

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

LAKE = Path("../data/lake/fixture/spark-warehouse")
def read(name):
    return pq.read_table(LAKE / name).to_pandas()

In [2]:
hist = read("labels_payments_history")
silver = read("payments_silver")[["txn_ref", "mcc", "event_ts", "is_refund"]]
fr = hist[hist.is_fraud].drop(columns=["event_ts"]).merge(silver, left_on="event_ref", right_on="txn_ref")
fr["known_from"] = pd.to_datetime(fr["__START_AT"].apply(lambda s: s["reported_at"]))
fr["event_month"] = fr.event_ts.dt.to_period("M")
month = fr.event_month.max() - 2
print("month under test:", month, "frauds in the slice:", len(fr))

month under test: 2019-06 frauds in the slice: 92


In [3]:
def leaky(frauds, month):
    window = (frauds.event_month < month) & (frauds.event_month >= month - 12)
    return int(window.sum())

def as_of(frauds, month):
    start = month.to_timestamp()
    window = (frauds.event_month < month) & (frauds.event_month >= month - 12) & (frauds.known_from < start)
    return int(window.sum())

planted = fr.iloc[[0]].copy()
planted["event_month"] = month - 1
planted["known_from"] = month.to_timestamp() + pd.Timedelta(seconds=1)
with_plant = pd.concat([fr, planted])
print("leaky count, before and after the plant:", leaky(fr, month), leaky(with_plant, month))
print("as of count, before and after the plant:", as_of(fr, month), as_of(with_plant, month))
assert as_of(fr, month) == as_of(with_plant, month)

leaky count, before and after the plant: 28 29


as of count, before and after the plant: 18 18


/tmp/ipykernel_32273/2664669922.py:12: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  planted["known_from"] = month.to_timestamp() + pd.Timedelta(seconds=1)


The leaky count moved by one: a label nobody knew at the start of the month changed a feature computed at the start of the month. The as of count did not. tests/transforms/test_transforms.py keeps the same plant against the Spark code (test_known_frauds_respects_the_system_time_clock).